In [1]:
from PIL import Image
import timm

img = Image.open('/Users/saptarshimallikthakur/Pictures/New Folder With Items/cat.jpeg')

model = timm.create_model(
    'mobilevitv2_075.cvnets_in1k',
    pretrained=True,
    features_only=True,
)
model = model.eval()

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

output = model(transforms(img).unsqueeze(0))[-1]
output.shape

/Users/saptarshimallikthakur/Pictures/VLM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.Size([1, 384, 8, 8])

In [2]:
data_config

{'input_size': (3, 256, 256),
 'interpolation': 'bicubic',
 'mean': (0.0, 0.0, 0.0),
 'std': (1.0, 1.0, 1.0),
 'crop_pct': 0.888,
 'crop_mode': 'center'}

In [3]:
output.view(output.size(0), output.size(1), -1).transpose(2,1).shape

torch.Size([1, 64, 384])

In [4]:
import torch
from torch import nn
import timm

class featureclassifier(nn.Module):
    def __init__(self,base_model_name='mobilevitv2_075.cvnets_in1k'):
        super(featureclassifier, self).__init__()
        self.base_model = timm.create_model(base_model_name,pretrained=True,features_only=True)
        self.base_model.eval()
        self.data_config = timm.data.resolve_model_data_config(self.base_model)
        self.transforms = timm.data.create_transform(**data_config, is_training=False)

        self.num_ori = 12
        self.num_shape = 40
        self.num_exp = 10
        self.last_channel = 384

        self.classifier_ori = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.last_channel, self.num_ori),
        )
        self.classifier_shape = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.last_channel, self.num_shape),
        )
        self.classifier_exp = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.last_channel, self.num_exp),
        )

    def forward(self,x):
        base_features = self.base_model(self.transforms(x).unsqueeze(0))[-1]
        base_features = nn.functional.adaptive_avg_pool2d(base_features, 1)
        base_features = base_features.reshape(base_features.shape[0], -1)

        x_ori = self.classifier_ori(base_features)
        x_shape = self.classifier_shape(base_features)
        x_exp = self.classifier_exp(base_features)

        print(x_ori.shape,x_shape.shape,x_exp.shape)

        x_3dmm = torch.cat((x_ori, x_shape, x_exp), dim=1)
        return x_3dmm, base_features


In [5]:
f = featureclassifier()
x,y = f(img)

torch.Size([1, 12]) torch.Size([1, 40]) torch.Size([1, 10])


In [6]:
x.shape,y.shape

(torch.Size([1, 62]), torch.Size([1, 384]))

In [7]:
import os.path as osp
import numpy as np
import pickle

def make_abs_path(d):
    return osp.join(osp.dirname(osp.realpath(__file__)), d)

def _get_suffix(filename):
    """a.jpg -> jpg"""
    pos = filename.rfind('.')
    if pos == -1:
        return ''
    return filename[pos + 1:]


def _load(fp):
    suffix = _get_suffix(fp)
    if suffix == 'npy':
        return np.load(fp)
    elif suffix == 'pkl':
        return pickle.load(open(fp, 'rb'))

class ParamsPack():
	"""Parameter package"""
	def __init__(self):
		try:
			d = '/Users/saptarshimallikthakur/Pictures/VLM/Synergynet/3dmm_data'
			self.keypoints = _load(osp.join(d, 'keypoints_sim.npy'))

			# PCA basis for shape, expression, texture
			self.w_shp = _load(osp.join(d, 'w_shp_sim.npy'))
			self.w_exp = _load(osp.join(d, 'w_exp_sim.npy'))
			# param_mean and param_std are used for re-whitening
			meta = _load(osp.join(d, 'param_whitening.pkl'))
			self.param_mean = meta.get('param_mean')
			self.param_std = meta.get('param_std')
			# mean values
			self.u_shp = _load(osp.join(d, 'u_shp.npy'))
			self.u_exp = _load(osp.join(d, 'u_exp.npy'))
			self.u = self.u_shp + self.u_exp
			self.w = np.concatenate((self.w_shp, self.w_exp), axis=1)
			# base vector for landmarks
			self.w_base = self.w[self.keypoints]
			self.w_norm = np.linalg.norm(self.w, axis=0)
			self.w_base_norm = np.linalg.norm(self.w_base, axis=0)
			self.u_base = self.u[self.keypoints].reshape(-1, 1)
			self.w_shp_base = self.w_shp[self.keypoints]
			self.w_exp_base = self.w_exp[self.keypoints]
			self.std_size = 120
			self.dim = self.w_shp.shape[0] // 3
		except:
			raise RuntimeError('Missing data')

In [8]:
pp = ParamsPack()

In [9]:
def parse_param_62(param):
	"""Work for only tensor"""
	p_ = param[:, :12].reshape(-1, 3, 4)
	p = p_[:, :, :3]
	offset = p_[:, :, -1].reshape(-1, 3, 1)
	alpha_shp = param[:, 12:52].reshape(-1, 40, 1)
	alpha_exp = param[:, 52:62].reshape(-1, 10, 1)
	return p, offset, alpha_shp, alpha_exp

p, offset, alpha_shp, alpha_exp = parse_param_62(x)

In [18]:
p.shape, offset.shape, alpha_shp.shape, alpha_exp.shape

(torch.Size([1, 3, 3]),
 torch.Size([1, 3, 1]),
 torch.Size([1, 40, 1]),
 torch.Size([1, 10, 1]))

In [14]:
import torch

# Convert pp components to tensors
pp_u = torch.from_numpy(pp.u).to(p.device)
pp_w_shp = torch.from_numpy(pp.w_shp).to(p.device)
pp_w_exp = torch.from_numpy(pp.w_exp).to(p.device)

In [ ]:
pp_u.shape,pp_w_shp.shape,pp_w_exp.shape

(torch.Size([159645, 1]), torch.Size([159645, 40]), torch.Size([159645, 10]))

In [22]:
vertex = p @ (pp_u + pp_w_shp @ alpha_shp + pp_w_exp @ alpha_exp).contiguous().view(-1, 53215, 3).transpose(1, 2) + offset

In [23]:
p.shape

torch.Size([1, 3, 3])

In [24]:
alpha_shp.shape

torch.Size([1, 40, 1])

In [25]:
k = pp_u + pp_w_shp @ alpha_shp + pp_w_exp @ alpha_exp
k.shape

torch.Size([1, 159645, 1])

In [26]:
k = k.contiguous().view(-1, 53215, 3).transpose(1, 2)
k.shape

torch.Size([1, 3, 53215])

In [27]:
vertex.shape

torch.Size([1, 3, 53215])

In [28]:
b = np.array([1, 3, 53215])
torch.tensor(b,dtype=torch.float)

tensor([1.0000e+00, 3.0000e+00, 5.3215e+04])

In [27]:
from utils import _load
_numpy_to_tensor = lambda x: torch.from_numpy(x)
_load_cpu = _load

params = _numpy_to_tensor(_load_cpu('/Users/saptarshimallikthakur/Pictures/VLM/Synergynet/3dmm_data/param_all_norm_v201.pkl'))

In [29]:
params[0].shape

torch.Size([102])

In [30]:
params.shape

torch.Size([636252, 102])